# Task 2: Vision-Language Models & Visual Grounding (ATML PA3)

**Course**: EE-5102 / CS-6304, Advanced Topics in Machine Learning, Fall 2026  
**Target Environment**: Google Colab with NVIDIA T4 GPU & Google Drive Persistent Storage  

This notebook provides the complete reproducible pipeline for **Task 2 only**:
1. Environment check & Google Drive mounting
2. Hardware preflight & autograd gradient flow verification
3. Deterministic data preparation (COCO captions, grounding pairs, natural VQAv2)
4. SigLIP visual feature extraction & FP16 caching (unloading vision encoder before training)
5. Three-stage training orchestrator (Stage 1 Caption Alignment -> Stage 2 Visual Instruction Tuning -> Stage 3 Natural VQA Specialization)
6. Crash-safe checkpoint resumption
7. Development evaluation across 12 scheduled checkpoints
8. Final test evaluation across 4 image conditions with 4,000 paired complete-pair bootstrap resamples
9. Qualitative aligned complementary-image inspection sheet
10. Automated generation of Tables T01–T12, Plots P01–P12, and Evidence Completeness Audit

## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

import os
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Install Locked Dependencies

In [ ]:
# Install task2 locked dependencies
%cd /content/drive/MyDrive/ATML_PA3/atml-pa3-FineTuning
!pip install -r task2/requirements.txt

## 3. Run Preflight Diagnostics & Autograd Smoke Test
Verifies that:
- Connector output shape and scale calibration buffers are initialized.
- Gradients flow through frozen Qwen into the connector.
- FP16 autocast and gradient checkpointing fit comfortably within T4 VRAM.

In [ ]:
!python -m task2.scripts.preflight --config task2/configs/task2.yaml

## 4. Deterministic Data Preparation & Disjointness Audit
Executes:
- 18,000 train captions (6,000 images x 3 captions) & 96 dev captions (1 caption/img)
- Grounding presence/counting pairs (dev: 24 pairs; test: 64 pairs)
- Natural VQAv2 complementary pairs (agreement >= 8/10; train: 1,536; dev: 32; test: 96)
- Generates split manifests, ID files, and Table T01 & T02

In [ ]:
!python -m task2.scripts.prepare_data --config task2/configs/task2.yaml

## 5. Resumable SigLIP Feature Caching
Extracts 196 patch features in FP16 for all selected images, verifies shapes `[196, 768]`, and releases the vision encoder from GPU memory before training.

In [ ]:
!python -m task2.scripts.cache_features --config task2/configs/task2.yaml

## 6. Three-Stage Training Orchestrator
- **Stage 1**: 2,250 updates, connector only (LR 1e-3), 2 passes over 18,000 captions
- **Stage 2**: 1,500 updates, connector (1e-4) + LoRA (2e-5), mixture (0.50 presence, 0.25 counting, 0.25 caption replay)
- **Stage 3**: 750 updates, connector (5e-5) + LoRA (1e-5), mixture (0.70 VQAv2, 0.10 presence, 0.10 counting, 0.10 caption replay)
Total: 4,500 updates, 72,000 example presentations. Checkpoints saved every 250 updates and at stage boundaries.

In [ ]:
# To train all 3 stages from scratch:
!python -m task2.scripts.train_stage --all --config task2/configs/task2.yaml

# If disconnected, resume from the latest verified checkpoint with:
# !python -m task2.scripts.train_stage --all --resume --config task2/configs/task2.yaml

## 7. Development Evaluation (12 Scheduled Checkpoints)
Computes caption reference loss, reference NLL across 3 conditions, and image-swap penalty $\Delta\text{NLL}$ across all 12 required checkpoints (0, 500, 1000, 1500, 2000, 2250, 2500, 3000, 3500, 3750, 4000, 4500).

In [ ]:
!python -m task2.scripts.evaluate_dev --all-scheduled --config task2/configs/task2.yaml

## 8. Final Evaluation & 4,000 Paired Bootstrap Analysis
Evaluates the final 4,500-step model on Grounding (64 pairs) and VQAv2 (96 pairs) under all 4 conditions (Correct, Complementary Swap, Mismatched, No Image) using greedy decoding, max 16 tokens, and 4,000 complete-pair bootstrap resamples.

In [ ]:
!python -m task2.scripts.evaluate_final --config task2/configs/task2.yaml --n-bootstrap 4000

## 9. Qualitative Inspection Sheet & Manifest

In [ ]:
!python -m task2.scripts.run_qualitative --config task2/configs/task2.yaml

## 10. Compile All Tables, Plots & Evidence Audit

In [ ]:
!python -m task2.scripts.build_results --config task2/configs/task2.yaml
!python -m task2.scripts.audit_evidence --config task2/configs/task2.yaml